# 01. 기초: 로그 전처리와 윈도우 라벨링

목표: NeuralLog가 왜 로그 파싱을 피하려 하는지 이해하고, raw log를 정규화한 뒤 sliding window label을 만드는 과정을 실습합니다.

실행 방법: 위에서부터 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. 예제 로그 만들기

BGL류 supercomputer 로그에서는 첫 문자가 `-`가 아니면 failure log로 취급할 수 있습니다. 아래 예제도 같은 관례를 따릅니다.

In [ ]:
raw_logs = [
    "- 2026-07-23 10:00:01 Node42 Disk sda1 read completed in 12ms",
    "- 2026-07-23 10:00:03 Node42 NetworkLink eth0 recovered",
    "E 2026-07-23 10:00:05 Node42 Disk sda1 read timeout sector 77881",
    "- 2026-07-23 10:00:08 Node17 UserLogin accepted from 10.0.0.5",
    "- 2026-07-23 10:00:12 Node17 CacheWarmup completed",
    "F 2026-07-23 10:00:14 Node17 KernelPanic reboot required code=0xDEAD",
    "- 2026-07-23 10:00:16 Node17 BootSequence started",
    "- 2026-07-23 10:00:20 Node42 Disk sda1 read completed in 10ms",
]

for line in raw_logs:
    label = 0 if line.startswith("-") else 1
    print(label, line)

## 2. NeuralLog식 전처리 흉내 내기

저장소 구현은 숫자와 특수문자를 제거하고 CamelCase를 나눕니다. 이는 파싱이 아니라 encoder가 처리하기 쉬운 텍스트 정규화입니다.

In [ ]:
import re
import string


def clean_log_message(line):
    # 첫 토큰은 BGL 스타일 label이라고 가정하고 제거합니다.
    content = line[line.find(" ") + 1 :]
    content = re.sub(r"\]|\[|\)|\(|=|,|;", " ", content)
    content = " ".join(word.lower() if word.isupper() else word for word in content.split())
    content = re.sub(r"([A-Z][a-z]+)", r" \1", re.sub(r"([A-Z]+)", r" \1", content))
    content = " ".join(word for word in content.split() if not re.search(r"\d", word))
    translator = str.maketrans("", "", string.punctuation)
    content = content.translate(translator)
    return " ".join(word.lower().strip() for word in content.split())


for line in raw_logs[:4]:
    print("raw:  ", line)
    print("clean:", clean_log_message(line))
    print()

## 3. 파싱 손실 관찰

템플릿 파서는 숫자와 변수 부분을 제거해 안정적인 event ID를 만들지만, 때로는 중요한 의미 차이까지 잃습니다. NeuralLog는 템플릿 ID 대신 정규화된 문장을 semantic encoder에 넣습니다.

In [ ]:
def naive_template(line):
    # 매우 단순한 파서입니다. 숫자, IP, 16진수, 장치명 일부를 변수로 바꿉니다.
    content = line[line.find(" ") + 1 :]
    content = re.sub(r"\b\d{4}-\d{2}-\d{2}\b", "<DATE>", content)
    content = re.sub(r"\b\d{2}:\d{2}:\d{2}\b", "<TIME>", content)
    content = re.sub(r"\b\d+(?:\.\d+){3}\b", "<IP>", content)
    content = re.sub(r"\b\d+\w*\b", "<NUM>", content)
    return content


pairs = [
    raw_logs[0],
    raw_logs[2],
    raw_logs[5],
]

for line in pairs:
    print("template:", naive_template(line))
    print("clean:   ", clean_log_message(line))
    print()

## 4. Sliding window 라벨 만들기

NeuralLog의 supercomputer loader는 window 안에 failure log가 하나라도 있으면 window label을 1로 둡니다.

In [ ]:
def make_windows(logs, window_size=3, step_size=2):
    windows = []
    labels = []
    for start in range(0, len(logs) - window_size + 1, step_size):
        chunk = logs[start : start + window_size]
        label = int(any(not line.startswith("-") for line in chunk))
        windows.append([clean_log_message(line) for line in chunk])
        labels.append(label)
    return windows, labels


windows, labels = make_windows(raw_logs)
for index, (window, label) in enumerate(zip(windows, labels)):
    print(f"window={index} label={label}")
    for message in window:
        print(" -", message)
    print()